# JEPX Scraping
**Scrape historical JEPX data**

## 1. Import relative library

In [29]:
import os
import time
import requests
import polars as pl
from bs4 import BeautifulSoup
from typing import Literal
from pathlib import Path
import numpy as np

## 2. Define constraint variable

In [30]:
STORAGE_PATH = r"/workspace/src/stg/data_lake/jepx"

PARENT_URL = "https://www.jepx.jp/electricpower/market-data/"

DATA_DICT = {
    "spot":"spot_summary",
    "intraday":"intraday",
    "forward":"forward",
    "transmission_rights":"transmission_rights",
    "baseload":"baseload",
    "fit_fip":"fit_fip"
}

DATA_DICT_KEYS = DATA_DICT.keys()

## 3. Define custom functions

In [31]:
def scrape_jepx_data(end_point:str, file_name:str, hist_year:str) -> bytes:
    """_summary_.

    Args:
        end_point (str): _description_
        file_name (str): _description_
        hist_year (str): _description_

    Returns:
        bytes: _description_
    """
    time_stamp = int(time.time() * 1000)

    url = f"https://www.jepx.jp/_download.php?timestamp={time_stamp}"

    headers = {
        "Content-Type" : "application/x-www-form-urlencoded",
        "Referer" : f"{PARENT_URL}/{end_point}/"
    }

    data = {
        "dir": file_name,
        "file": f"{file_name}_{hist_year}.csv"
    }

    response = requests.post(url, headers=headers, data=data)

    return response.content

In [32]:
def save_csv_as_parquet(csv_bytes:bytes, output_path: Path | str, *, compression: Literal["zstd","snappy","gzip","lz4","uncompressed"]="zstd", encoding:str = "cp932") -> Path:
    """_summary_

    Args:
        csv_bytes (bytes): _description_
        output_path (Path | str): _description_
        compression (Literal["zstd","snappy","gzip","lz4","uncompressed"], optional): _description_. Defaults to "zstd".
        encoding (str, optional): _description_. Defaults to "cp932".

    Returns:
    """

    output_path = Path(output_path)

    df = pl.read_csv(csv_bytes, encoding=encoding)

    df.write_parquet(output_path, compression=compression)

    print(f"✓ Saved {len(df):,} rows to {output_path}")
    print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

    return output_path

## 4. Scrape JEPX data

In [33]:
hist_year_range = range(2005,2026)

for data_key in DATA_DICT_KEYS:
    end_point = DATA_DICT[data_key]
    for hist_year in hist_year_range:
        print(f"Scraping {data_key} data for year {hist_year}...")
        try:
            time.sleep(np.random.uniform(1,5))
            csv_bytes = scrape_jepx_data(data_key, end_point, str(hist_year))

            output_dir = Path(STORAGE_PATH) / data_key

            output_path = output_dir / f"{end_point}_{hist_year}.parquet"

            save_csv_as_parquet(csv_bytes, output_path)
        except Exception as e:
            print(f"✗ Failed to scrape {data_key} data for year {hist_year}: {e}")
            continue

Scraping spot data for year 2005...
✓ Saved 17,472 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2005.parquet
  File size: 0.26 MB
Scraping spot data for year 2006...
✓ Saved 17,520 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2006.parquet
  File size: 0.28 MB
Scraping spot data for year 2007...
✓ Saved 17,568 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2007.parquet
  File size: 0.33 MB
Scraping spot data for year 2008...
✓ Saved 17,520 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2008.parquet
  File size: 0.34 MB
Scraping spot data for year 2009...
✓ Saved 17,520 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2009.parquet
  File size: 0.31 MB
Scraping spot data for year 2010...
✓ Saved 17,520 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2010.parquet
  File size: 0.36 MB
Scraping spot data for year 2011...
✓ Saved 17,568 rows to /workspace/src/stg/data_lake/jepx/spot/spot_summary_2011.parquet
  File

In [34]:
def scrape_jepx_nf_data(hist_year:str, *, file_name:str="nf_summary") -> bytes:
    """_summary_.

    Args:
        end_point (str): _description_
        file_name (str): _description_
        hist_year (str): _description_

    Returns:
        bytes: _description_
    """
    time_stamp = int(time.time() * 1000)

    NF_PARENT_URL = "https://www.jepx.jp/nonfossil/market-data/"

    url = f"https://www.jepx.jp/_download.php?timestamp={time_stamp}"

    headers = {
        "Content-Type" : "application/x-www-form-urlencoded",
        "Referer" : f"{NF_PARENT_URL}/"
    }

    data = {
        "dir": file_name,
        "file": f"{file_name}_{hist_year}.csv"
    }

    response = requests.post(url, headers=headers, data=data)

    return response.content

In [35]:
hist_year_range = range(2017,2026)


for hist_year in hist_year_range:
    print(f"Scraping nonfossil data for year {hist_year}...")
    try:
        time.sleep(np.random.uniform(1,5))

        csv_bytes = scrape_jepx_nf_data(str(hist_year))

        output_dir = Path(STORAGE_PATH) / "nonfossil"

        output_path = output_dir / f"nf_summary_{hist_year}.parquet"

        save_csv_as_parquet(csv_bytes, output_path)
    except Exception as e:
        print(f"✗ Failed to scrape {data_key} data for year {hist_year}: {e}")
        continue

Scraping nonfossil data for year 2017...
✓ Saved 3 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2017.parquet
  File size: 0.00 MB
Scraping nonfossil data for year 2018...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2018.parquet
  File size: 0.00 MB
Scraping nonfossil data for year 2019...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2019.parquet
  File size: 0.00 MB
Scraping nonfossil data for year 2020...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2020.parquet
  File size: 0.00 MB
Scraping nonfossil data for year 2021...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2021.parquet
  File size: 0.00 MB
Scraping nonfossil data for year 2022...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_summary_2022.parquet
  File size: 0.01 MB
Scraping nonfossil data for year 2023...
✓ Saved 12 rows to /workspace/src/stg/data_lake/jepx/nonfossil/nf_